
#  PROYECTO: PREDICCIÓN DE POPULARIDAD DE PELÍCULAS

# 📌 Introducción

El objetivo de este proyecto es analizar diferentes características de películas para identificar cuáles factores influyen en el éxito comercial de una producción cinematográfica.

Actualmente, la industria del cine genera enormes cantidades de datos relacionados con presupuestos, popularidad, géneros, calificaciones y ganancias. Analizar esta información permite comprender patrones importantes dentro del mercado cinematográfico y construir modelos capaces de predecir el desempeño de futuras películas.

El análisis resulta interesante porque combinamos técnicas de ciencia de datos y aprendizaje automático para resolver un problema real: estimar el éxito de una película utilizando información histórica.

------------------------------------------

# 🎯 Objetivo del Proyecto

Construir un modelo predictivo capaz de identificar y clasificar películas exitosas y no exitosas utilizando variables relacionadas con presupuesto, popularidad, duración, votos y otras características disponibles en el dataset.

---

# 🧠 Tipo de Análisis

El enfoque utilizado corresponde a Aprendizaje Supervisado (Predicción).

La variable objetivo será:

- **1 → Película exitosa**
- **0 → Película no exitosa**

---

# 📂 Dataset Utilizado

Se utilizará el dataset **The Movies Dataset**, disponible públicamente en Kaggle.

Link:
https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset








#  SECCIÓN 1 — INTRODUCCIÓN, CARGA Y EXPLORACIÓN

In [ ]:
import pandas as pd
import numpy as np
import ast
import warnings
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")

print("""
╔══════════════════════════════════════════════════════════╗
║   PREGUNTA: ¿Qué características de una película         ║
║   predicen mejor su popularidad en TMDB?                 ║
║                                                          ║
║   ENFOQUE: Regresión supervisada + SHAP                  ║
╚══════════════════════════════════════════════════════════╝

POR QUÉ ES INTERESANTE:
  • La popularidad determina visibilidad en plataformas de streaming.
  • Permite a productoras tomar decisiones basadas en datos.
  • Es un problema con variables mixtas: numéricas + categóricas.
""")


╔══════════════════════════════════════════════════════════╗
║   PREGUNTA: ¿Qué características de una película         ║
║   predicen mejor su popularidad en TMDB?                 ║
║                                                          ║
║   ENFOQUE: Regresión supervisada + SHAP                  ║
╚══════════════════════════════════════════════════════════╝

POR QUÉ ES INTERESANTE:
  • La popularidad determina visibilidad en plataformas de streaming.
  • Permite a productoras tomar decisiones basadas en datos.
  • Es un problema con variables mixtas: numéricas + categóricas.



In [ ]:
#1.1 Carga ───────────────────────────────────────────────
df = pd.read_csv("movies_metadata.csv", low_memory=False)
print(f"Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")

#1.2  Limpieza ────────────────────────────────────────────
movies = df[["title","budget","revenue","runtime",
             "vote_average","vote_count","popularity","genres"]].copy()

for col in ["budget","revenue","popularity"]:
    movies[col] = pd.to_numeric(movies[col], errors="coerce")

def parse_genres(g):
    try:
        return [x["name"] for x in ast.literal_eval(g)] if pd.notna(g) else []
    except Exception:
        return []

movies["genre_list"] = movies["genres"].apply(parse_genres)

TOP_GENRES = ["Drama","Comedy","Thriller","Action",
              "Romance","Horror","Crime","Adventure"]
for g in TOP_GENRES:
    movies[g] = movies["genre_list"].apply(lambda lst: int(g in lst))

movies = movies.dropna(subset=["popularity"])
for col in ["budget","revenue"]:
    movies[col] = movies[col].replace(0, np.nan)
    movies[col].fillna(movies[col].median(), inplace=True)
movies["runtime"].fillna(movies["runtime"].median(), inplace=True)
movies["vote_average"].fillna(movies["vote_average"].median(), inplace=True)
movies["vote_count"].fillna(movies["vote_count"].median(), inplace=True)
movies.drop_duplicates(subset=["title"], inplace=True)

movies["log_budget"]     = np.log1p(movies["budget"])
movies["log_revenue"]    = np.log1p(movies["revenue"])
movies["log_votes"]      = np.log1p(movies["vote_count"])
movies["log_popularity"] = np.log1p(movies["popularity"])

print(f"Dataset limpio: {movies.shape[0]:,} filas\n")


Dataset cargado: 45,466 filas × 24 columnas
Dataset limpio: 42,277 filas



In [ ]:
# ────────────────────────────────────────────────────────────
#  VIZ 1 — Distribuciones principales (4 paneles)
# ────────────────────────────────────────────────────────────
fig1 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Calificación promedio (vote_average)",
        "Duración de la película (minutos)",
        "Volumen de votos — escala log",
        "Popularidad TMDB — escala log"
    ]
)
palette = ["#00B4D8","#F77F00","#06D6A0","#EF476F"]
cols_plot = ["vote_average","runtime","log_votes","log_popularity"]
positions = [(1,1),(1,2),(2,1),(2,2)]

for col, (r,c), color in zip(cols_plot, positions, palette):
    vals = movies[col].dropna()
    fig1.add_trace(
        go.Histogram(x=vals, nbinsx=50, marker_color=color,
                     name=col, showlegend=False,
                     hovertemplate=f"<b>{col}</b><br>Valor: %{{x:.2f}}<br>Count: %{{y}}<extra></extra>"),
        row=r, col=c
    )

fig1.update_layout(
    title=dict(text="<b>DISTRIBUCIONES PRINCIPALES</b><br>"
               "<sup>Cada histograma es interactivo — zoom, hover y selección habilitados</sup>",
               font=dict(size=16)),
    height=550, template="plotly_dark",
    paper_bgcolor="#0D1117", plot_bgcolor="#161B22",
    font=dict(color="#C9D1D9")
)
fig1.show()
# fig1.write_html("viz1_distribuciones.html")  # ← descomentar para exportar

print("""
📊 INTERPRETACIÓN — VIZ 1:
  • vote_average: distribución casi normal centrada en ~6.5.
    Pocas películas <4 o >8; la mayoría es "aceptable".
  • runtime: pico claro en 90-110 min (formato comercial estándar).
    Cola larga a la derecha: documentales y épicas.
  • log_votes: sesgado a la izquierda → la mayoría tiene pocos votos;
    sólo un puñado de blockbusters acumula millones.
  • log_popularity: similar a log_votes; confirma que la popularidad
    está concentrada en pocas películas de alto impacto.
""")



📊 INTERPRETACIÓN — VIZ 1:
  • vote_average: distribución casi normal centrada en ~6.5.
    Pocas películas <4 o >8; la mayoría es "aceptable".
  • runtime: pico claro en 90-110 min (formato comercial estándar).
    Cola larga a la derecha: documentales y épicas.
  • log_votes: sesgado a la izquierda → la mayoría tiene pocos votos;
    sólo un puñado de blockbusters acumula millones.
  • log_popularity: similar a log_votes; confirma que la popularidad
    está concentrada en pocas películas de alto impacto.



In [ ]:
# ────────────────────────────────────────────────────────────
#  VIZ 2 — Correlaciones con la popularidad
# ────────────────────────────────────────────────────────────
num_cols = ["log_budget","log_revenue","runtime",
            "vote_average","log_votes","log_popularity"]
corr = movies[num_cols].corr()

fig2 = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="<b>MATRIZ DE CORRELACIONES</b><br>"
          "<sup>Hover para valor exacto — escala −1 a +1</sup>",
    labels=dict(color="Correlación")
)
fig2.update_layout(
    height=500, template="plotly_dark",
    paper_bgcolor="#0D1117",
    font=dict(color="#C9D1D9", size=13),
    coloraxis_colorbar=dict(title="r")
)
fig2.show()
# fig2.write_html("viz2_correlaciones.html")

print("""
📊 INTERPRETACIÓN — VIZ 2:
  • log_votes ↔ log_popularity: correlación más alta (~0.85).
    El número de personas que vota es el mejor proxy de popularidad.
  • log_revenue ↔ log_popularity: ~0.60. Taquilla alta = mayor
    difusión y por ende más interacciones en TMDB.
  • log_budget ↔ log_popularity: ~0.45. Presupuesto ayuda, pero
    menos que el resultado de taquilla.
  • vote_average: correlación baja (~0.15). La calidad percibida
    importa poco para la popularidad bruta.
""")




📊 INTERPRETACIÓN — VIZ 2:
  • log_votes ↔ log_popularity: correlación más alta (~0.85).
    El número de personas que vota es el mejor proxy de popularidad.
  • log_revenue ↔ log_popularity: ~0.60. Taquilla alta = mayor
    difusión y por ende más interacciones en TMDB.
  • log_budget ↔ log_popularity: ~0.45. Presupuesto ayuda, pero
    menos que el resultado de taquilla.
  • vote_average: correlación baja (~0.15). La calidad percibida
    importa poco para la popularidad bruta.



In [ ]:
# ────────────────────────────────────────────────────────────
#  VIZ 3 — Popularidad por género (boxplot interactivo)
# ────────────────────────────────────────────────────────────
genre_rows = []
for g in TOP_GENRES:
    sub = movies[movies[g] == 1]["log_popularity"]
    for v in sub:
        genre_rows.append({"genre": g, "log_popularity": v})
genre_df = pd.DataFrame(genre_rows)

medians = (genre_df.groupby("genre")["log_popularity"]
           .median().sort_values(ascending=False).index.tolist())

fig3 = px.box(
    genre_df, x="genre", y="log_popularity",
    category_orders={"genre": medians},
    color="genre",
    color_discrete_sequence=px.colors.qualitative.Bold,
    title="<b>DISTRIBUCIÓN DE POPULARIDAD POR GÉNERO</b><br>"
          "<sup>Ordenado por mediana — click en leyenda para filtrar</sup>",
    labels={"log_popularity": "log(popularidad)", "genre": "Género"}
)
fig3.update_layout(
    height=480, template="plotly_dark",
    paper_bgcolor="#0D1117", plot_bgcolor="#161B22",
    font=dict(color="#C9D1D9"),
    showlegend=False
)
fig3.show()
# fig3.write_html("viz3_generos.html")

print("""
📊 INTERPRETACIÓN — VIZ 3:
  • Action y Adventure tienen las medianas más altas: estos géneros
    tienen lanzamientos masivos con campañas globales de marketing.
  • Horror muestra alta varianza: hay éxitos de culto y muchas
    producciones de bajo presupuesto que pasan desapercibidas.
  • Drama tiene la mediana más baja pese a ser el género más frecuente.
    La abundancia diluye la visibilidad individual de cada película.
""")




📊 INTERPRETACIÓN — VIZ 3:
  • Action y Adventure tienen las medianas más altas: estos géneros
    tienen lanzamientos masivos con campañas globales de marketing.
  • Horror muestra alta varianza: hay éxitos de culto y muchas
    producciones de bajo presupuesto que pasan desapercibidas.
  • Drama tiene la mediana más baja pese a ser el género más frecuente.
    La abundancia diluye la visibilidad individual de cada película.




#  SECCIÓN 2 — MODELOS PREDICTIVOS


In [ ]:

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

FEATURES = (["log_budget","log_revenue","runtime",
              "vote_average","log_votes"] + TOP_GENRES)
TARGET   = "log_popularity"

X = movies[FEATURES]
y = movies[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

pipe_ridge = Pipeline([("sc", StandardScaler()),
                       ("m",  Ridge(alpha=10))])
pipe_rf    = Pipeline([("sc", StandardScaler()),
                       ("m",  RandomForestRegressor(
                           n_estimators=300, max_depth=12,
                           n_jobs=-1, random_state=42))])
pipe_xgb   = Pipeline([("sc", StandardScaler()),
                       ("m",  XGBRegressor(
                           n_estimators=400, max_depth=6,
                           learning_rate=0.05, subsample=0.8,
                           colsample_bytree=0.8, random_state=42,
                           verbosity=0))])

MODELS = {"Ridge": pipe_ridge,
          "Random Forest": pipe_rf,
          "XGBoost": pipe_xgb}

resultados = []
for name, pipe in MODELS.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    mae  = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2   = r2_score(y_test, pred)
    cv   = cross_val_score(pipe, X_train, y_train, cv=5, scoring="r2").mean()
    resultados.append({"model": name, "pipe": pipe, "pred": pred,
                       "mae": mae, "rmse": rmse, "r2": r2, "cv_r2": cv})
    print(f"  {name:15s} | MAE={mae:.4f} | RMSE={rmse:.4f} | R²={r2:.4f} | CV-R²={cv:.4f}")

mejor = max(resultados, key=lambda r: r["r2"])
print(f"\n✅  Mejor modelo: {mejor['model']}  (R²={mejor['r2']:.4f})")

Train: (33821, 13)  |  Test: (8456, 13)
  Ridge           | MAE=0.2602 | RMSE=0.3605 | R²=0.8000 | CV-R²=0.8007
  Random Forest   | MAE=0.2467 | RMSE=0.3392 | R²=0.8229 | CV-R²=0.8220
  XGBoost         | MAE=0.2449 | RMSE=0.3365 | R²=0.8257 | CV-R²=0.8248

✅  Mejor modelo: XGBoost  (R²=0.8257)


In [ ]:
# ────────────────────────────────────────────────────────────
#  VIZ 4 — Comparación de métricas (radar chart)
# ────────────────────────────────────────────────────────────
res_df = pd.DataFrame([{k: v for k, v in r.items()
                         if k not in ("pipe","pred")} for r in resultados])

# Normalizar para radar (0-1, mayor=mejor para R²; menor=mejor para MAE/RMSE → invertir)
res_norm = res_df.copy()
for m in ["mae","rmse"]:
    mx = res_norm[m].max()
    res_norm[m] = 1 - (res_norm[m] / mx)
for m in ["r2","cv_r2"]:
    mn, mx = res_norm[m].min(), res_norm[m].max()
    res_norm[m] = (res_norm[m] - mn) / (mx - mn + 1e-9)

cats = ["MAE (inv)","RMSE (inv)","R²","CV-R²"]
colors = ["#00B4D8","#F77F00","#06D6A0"]

fig4 = go.Figure()
for i, row in res_norm.iterrows():
    vals = [row["mae"], row["rmse"], row["r2"], row["cv_r2"]]
    vals += [vals[0]]
    fig4.add_trace(go.Scatterpolar(
        r=vals, theta=cats+[cats[0]],
        fill="toself", name=row["model"],
        line_color=colors[i], opacity=0.7
    ))

fig4.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1],
               gridcolor="#30363D", linecolor="#30363D"),
               angularaxis=dict(gridcolor="#30363D", linecolor="#30363D"),
               bgcolor="#161B22"),
    title=dict(text="<b>COMPARACIÓN DE MODELOS — RADAR</b><br>"
               "<sup>Mayor área = mejor desempeño global</sup>",
               font=dict(size=15)),
    template="plotly_dark",
    paper_bgcolor="#0D1117",
    font=dict(color="#C9D1D9"),
    height=480,
    legend=dict(bgcolor="#161B22", bordercolor="#30363D")
)
fig4.show()
# fig4.write_html("viz4_radar_modelos.html")

print("""
📊 INTERPRETACIÓN — VIZ 4:
  • XGBoost domina en todas las métricas; su área cubre la mayor
    superficie del radar, especialmente en R² y CV-R².
  • Random Forest es sólido pero ligeramente inferior en generalización.
  • Ridge (lineal) tiene buen CV-R² pero peor RMSE, lo que indica
    que falla en películas de popularidad extrema (no-linealidades).
""")


📊 INTERPRETACIÓN — VIZ 4:
  • XGBoost domina en todas las métricas; su área cubre la mayor
    superficie del radar, especialmente en R² y CV-R².
  • Random Forest es sólido pero ligeramente inferior en generalización.
  • Ridge (lineal) tiene buen CV-R² pero peor RMSE, lo que indica
    que falla en películas de popularidad extrema (no-linealidades).



In [ ]:
# ────────────────────────────────────────────────────────────
#  VIZ 5 — Real vs Predicho + residuos (mejor modelo)
# ────────────────────────────────────────────────────────────
pred_mejor = mejor["pred"]
residuos   = y_test.values - pred_mejor

fig5 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Real vs Predicho", "Distribución de Residuos"]
)

fig5.add_trace(
    go.Scatter(
        x=y_test, y=pred_mejor, mode="markers",
        marker=dict(color=residuos, colorscale="RdYlGn",
                    size=4, opacity=0.6,
                    colorbar=dict(title="Residuo", x=0.45)),
        hovertemplate="Real: %{x:.2f}<br>Predicho: %{y:.2f}<extra></extra>",
        name="películas"
    ), row=1, col=1
)
lims = [float(min(y_test.min(), pred_mejor.min())),
        float(max(y_test.max(), pred_mejor.max()))]
fig5.add_trace(
    go.Scatter(x=lims, y=lims, mode="lines",
               line=dict(color="#EF476F", dash="dash", width=2),
               name="ideal", showlegend=False),
    row=1, col=1
)

fig5.add_trace(
    go.Histogram(x=residuos, nbinsx=60,
                 marker_color="#00B4D8", opacity=0.8,
                 hovertemplate="Residuo: %{x:.3f}<br>Count: %{y}<extra></extra>",
                 name="residuos"),
    row=1, col=2
)
fig5.add_vline(x=0, line_dash="dash", line_color="#EF476F", row=1, col=2)

fig5.update_xaxes(title_text="log(popularidad) real", row=1, col=1)
fig5.update_yaxes(title_text="log(popularidad) predicha", row=1, col=1)
fig5.update_xaxes(title_text="Residuo", row=1, col=2)
fig5.update_yaxes(title_text="Frecuencia", row=1, col=2)

fig5.update_layout(
    title=dict(text=f"<b>DESEMPEÑO DEL MEJOR MODELO — {mejor['model']}</b><br>"
               "<sup>Color = magnitud del error | Hover para detalles</sup>",
               font=dict(size=15)),
    height=460, template="plotly_dark",
    paper_bgcolor="#0D1117", plot_bgcolor="#161B22",
    font=dict(color="#C9D1D9"), showlegend=False
)
fig5.show()
# fig5.write_html("viz5_residuos.html")

print(f"""
📊 INTERPRETACIÓN — VIZ 5:
  • Los puntos se alinean bien sobre la diagonal ideal (R²={mejor['r2']:.3f}).
  • Los residuos se distribuyen simétricamente alrededor de 0 → sin
    sesgo sistemático.
  • Los puntos más alejados (rojo) corresponden a películas virales
    atípicas (ej. franquicias en semana de estreno) que el modelo
    sub-predice; es un límite estructural de los datos disponibles.
""")



📊 INTERPRETACIÓN — VIZ 5:
  • Los puntos se alinean bien sobre la diagonal ideal (R²=0.828).
  • Los residuos se distribuyen simétricamente alrededor de 0 → sin
    sesgo sistemático.
  • Los puntos más alejados (rojo) corresponden a películas virales
    atípicas (ej. franquicias en semana de estreno) que el modelo
    sub-predice; es un límite estructural de los datos disponibles.




#  SECCIÓN 3 — SHAP + VISUALIZACIONES INTERACTIVAS


In [ ]:
import shap

print("\n=== Calculando valores SHAP ===")

best_pipe  = mejor["pipe"]
best_model = best_pipe.named_steps["m"]

X_test_scaled = pd.DataFrame(
    best_pipe.named_steps["sc"].transform(X_test),
    columns=FEATURES
)

# Muestrear para velocidad
sample_idx = np.random.choice(len(X_test_scaled), size=min(800, len(X_test_scaled)), replace=False)
X_shap = X_test_scaled.iloc[sample_idx].reset_index(drop=True)

if mejor["model"] == "Ridge":
    explainer   = shap.LinearExplainer(best_model, X_shap)
    shap_values = explainer.shap_values(X_shap)
else:
    explainer   = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_shap)

# ── Importancia media ────────────────────────────────────────
mean_abs    = np.abs(shap_values).mean(axis=0)
importancia = pd.Series(mean_abs, index=FEATURES).sort_values(ascending=False)

# ── VIZ 6 — SHAP Bar Plot interactivo ───────────────────────
fig6 = go.Figure(go.Bar(
    x=importancia.values[::-1],
    y=importancia.index[::-1],
    orientation="h",
    marker=dict(
        color=importancia.values[::-1],
        colorscale="Viridis",
        showscale=True,
        colorbar=dict(title="|SHAP|")
    ),
    hovertemplate="<b>%{y}</b><br>Importancia media: %{x:.4f}<extra></extra>"
))
fig6.update_layout(
    title=dict(text="<b>SHAP — IMPORTANCIA MEDIA DE VARIABLES</b><br>"
               "<sup>Mayor valor = mayor impacto promedio en la predicción</sup>",
               font=dict(size=15)),
    xaxis_title="mean |SHAP value|",
    yaxis_title="Variable",
    height=480, template="plotly_dark",
    paper_bgcolor="#0D1117", plot_bgcolor="#161B22",
    font=dict(color="#C9D1D9")
)
fig6.show()
# fig6.write_html("viz6_shap_bar.html")

print("""
📊 INTERPRETACIÓN — VIZ 6:
  • log_votes lidera con gran margen: el engagement (votar) es la
    señal más fuerte de popularidad — causalidad bidireccional.
  • log_revenue en segundo lugar: la taquilla refleja alcance global
    de distribución y marketing.
  • vote_average sorprende por su bajo impacto: ser "buena" no garantiza
    ser "popular" — confirma que la calidad ≠ visibilidad.
  • Los géneros tienen impacto modesto pero colectivamente suman;
    Action supera consistentemente a Drama.
""")


=== Calculando valores SHAP ===



📊 INTERPRETACIÓN — VIZ 6:
  • log_votes lidera con gran margen: el engagement (votar) es la
    señal más fuerte de popularidad — causalidad bidireccional.
  • log_revenue en segundo lugar: la taquilla refleja alcance global
    de distribución y marketing.
  • vote_average sorprende por su bajo impacto: ser "buena" no garantiza
    ser "popular" — confirma que la calidad ≠ visibilidad.
  • Los géneros tienen impacto modesto pero colectivamente suman;
    Action supera consistentemente a Drama.



In [ ]:
# ── VIZ 7 — SHAP Beeswarm interactivo (Plotly) ──────────────
shap_df = pd.DataFrame(shap_values, columns=FEATURES)
feat_order = importancia.index.tolist()[:8]  # top 8

fig7 = go.Figure()
color_scale = px.colors.sequential.Plasma

for i, feat in enumerate(feat_order[::-1]):
    sv   = shap_df[feat].values
    fv   = X_shap[feat].values
    # Normalizar valor de feature a [0,1] para color
    fv_n = (fv - fv.min()) / (fv.max() - fv.min() + 1e-9)
    colors_hex = [
        f"rgb({int(255*(1-v))},{int(120*v)},{int(200*v)})"
        for v in fv_n
    ]
    # Añadir jitter en y para el beeswarm
    jitter = np.random.uniform(-0.35, 0.35, len(sv))
    fig7.add_trace(go.Scatter(
        x=sv,
        y=np.full(len(sv), i) + jitter,
        mode="markers",
        marker=dict(color=fv_n, colorscale="Plasma",
                    size=4, opacity=0.55,
                    showscale=(i == len(feat_order)-1),
                    colorbar=dict(title="Valor<br>feature", x=1.02)),
        name=feat,
        hovertemplate=(f"<b>{feat}</b><br>"
                       "SHAP: %{x:.4f}<br>"
                       "Feature val: %{customdata:.3f}<extra></extra>"),
        customdata=fv,
        showlegend=False
    ))

fig7.update_layout(
    title=dict(text="<b>SHAP BEESWARM — EFECTO DE CADA VARIABLE</b><br>"
               "<sup>Color = valor de la feature | Posición X = impacto en predicción | Hover para detalles</sup>",
               font=dict(size=15)),
    xaxis_title="Valor SHAP (impacto en log_popularity)",
    yaxis=dict(
        tickmode="array",
        tickvals=list(range(len(feat_order))),
        ticktext=feat_order[::-1]
    ),
    height=520, template="plotly_dark",
    paper_bgcolor="#0D1117", plot_bgcolor="#161B22",
    font=dict(color="#C9D1D9")
)
fig7.show()
# fig7.write_html("viz7_shap_beeswarm.html")

print("""
📊 INTERPRETACIÓN — VIZ 7 (Beeswarm):
  • log_votes: valores altos (amarillo) → SHAP positivo fuerte.
    Pocas películas con millones de votos jalan la popularidad al alza.
  • log_revenue: mismo patrón; alta recaudación = fuerte empuje positivo.
  • vote_average: nube centrada en 0 → impacto casi nulo independiente
    del valor; confirma VIZ 6.
  • Action y Adventure: cuando están presentes (valor=1, amarillo)
    contribuyen positivamente; su ausencia es neutra.
""")



📊 INTERPRETACIÓN — VIZ 7 (Beeswarm):
  • log_votes: valores altos (amarillo) → SHAP positivo fuerte.
    Pocas películas con millones de votos jalan la popularidad al alza.
  • log_revenue: mismo patrón; alta recaudación = fuerte empuje positivo.
  • vote_average: nube centrada en 0 → impacto casi nulo independiente
    del valor; confirma VIZ 6.
  • Action y Adventure: cuando están presentes (valor=1, amarillo)
    contribuyen positivamente; su ausencia es neutra.



In [ ]:
# ── VIZ 8 — SHAP Dependence Plot interactivo ────────────────
top_feat   = importancia.index[0]   # log_votes
inter_feat = importancia.index[1]   # log_revenue

sv_top  = shap_df[top_feat].values
fv_top  = X_shap[top_feat].values
fv_inter = X_shap[inter_feat].values

fig8 = px.scatter(
    x=fv_top, y=sv_top,
    color=fv_inter,
    color_continuous_scale="Turbo",
    labels={"x": top_feat, "y": f"SHAP ({top_feat})",
            "color": inter_feat},
    title=f"<b>SHAP DEPENDENCE PLOT — {top_feat}</b><br>"
          f"<sup>Color = {inter_feat} | Revela interacciones entre variables</sup>",
    template="plotly_dark",
    opacity=0.6
)
fig8.update_traces(marker_size=5,
    hovertemplate=(f"{top_feat}: %{{x:.3f}}<br>"
                   f"SHAP: %{{y:.4f}}<br>"
                   f"{inter_feat}: %{{marker.color:.3f}}<extra></extra>"))
fig8.update_layout(
    height=460,
    paper_bgcolor="#0D1117", plot_bgcolor="#161B22",
    font=dict(color="#C9D1D9"),
    coloraxis_colorbar=dict(title=inter_feat)
)
fig8.show()
# fig8.write_html("viz8_shap_dependence.html")

print(f"""
📊 INTERPRETACIÓN — VIZ 8 (Dependence Plot):
  • Relación positiva clara: más log_votes → mayor SHAP → más popularidad.
  • El color (log_revenue) muestra interacción: puntos amarillos
    (alta recaudación) tienden a estar en la parte superior del SHAP,
    indicando que el efecto de los votos se amplifica cuando la
    película también recauda mucho.
  • La nube se estrecha en valores bajos de log_votes, sugiriendo que
    películas con pocos votos tienen comportamiento más predecible.
""")

# Exportar valores SHAP para dashboard externo
shap_export = shap_df[feat_order].copy()
shap_export.columns = [f"shap_{c}" for c in shap_export.columns]
feat_export = X_shap[feat_order].copy()
combined = pd.concat([feat_export, shap_export], axis=1)
combined["pred_popularity"] = mejor["pipe"].predict(X_test)[sample_idx]
combined.to_csv("shap_data_export.csv", index=False)
print("\n✅  Datos SHAP exportados a shap_data_export.csv")
print("    → Úsalos en el dashboard HTML interactivo (proyecto_dashboard.html)\n")



📊 INTERPRETACIÓN — VIZ 8 (Dependence Plot):
  • Relación positiva clara: más log_votes → mayor SHAP → más popularidad.
  • El color (log_revenue) muestra interacción: puntos amarillos
    (alta recaudación) tienden a estar en la parte superior del SHAP,
    indicando que el efecto de los votos se amplifica cuando la
    película también recauda mucho.
  • La nube se estrecha en valores bajos de log_votes, sugiriendo que
    películas con pocos votos tienen comportamiento más predecible.


✅  Datos SHAP exportados a shap_data_export.csv
    → Úsalos en el dashboard HTML interactivo (proyecto_dashboard.html)



In [ ]:

# ============================================================
#  MINI APP — PREDICTOR DE POPULARIDAD DE PELÍCULAS (TMDB)
#  Pregunta: ¿Qué características predicen mejor la popularidad?
#
#  Ejecución:
#    pip install gradio shap xgboost scikit-learn pandas numpy plotly
#    python app_gradio.py
# ============================================================

import pandas as pd
import numpy as np
import ast
import warnings
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import shap
import gradio as gr
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import os

warnings.filterwarnings("ignore")

In [ ]:
# ── Constantes ───────────────────────────────────────────────
TOP_GENRES = ["Drama","Comedy","Thriller","Action",
              "Romance","Horror","Crime","Adventure"]
FEATURES   = (["log_budget","log_revenue","runtime",
                "vote_average","log_votes"] + TOP_GENRES)
TARGET     = "log_popularity"

NIVEL_LABELS = ["Muy baja","Baja","Media","Alta","Muy alta","Viral"]
NIVEL_EMOJIS = ["🥶","😐","🙂","🔥","🚀","🌟"]

In [ ]:
# ════════════════════════════════════════════════════════════
#  ENTRENAMIENTO (se ejecuta al iniciar la app)
# ════════════════════════════════════════════════════════════

def entrenar_modelo(csv_path="movies_metadata.csv"):
    df = pd.read_csv(csv_path, low_memory=False)
    movies = df[["title","budget","revenue","runtime",
                 "vote_average","vote_count","popularity","genres"]].copy()

    for col in ["budget","revenue","popularity"]:
        movies[col] = pd.to_numeric(movies[col], errors="coerce")

    def parse_genres(g):
        try:
            return [x["name"] for x in ast.literal_eval(g)] if pd.notna(g) else []
        except Exception:
            return []

    movies["genre_list"] = movies["genres"].apply(parse_genres)
    for g in TOP_GENRES:
        movies[g] = movies["genre_list"].apply(lambda lst: int(g in lst))

    movies = movies.dropna(subset=["popularity"])
    for col in ["budget","revenue"]:
        movies[col] = movies[col].replace(0, np.nan)
        movies[col].fillna(movies[col].median(), inplace=True)
    movies["runtime"].fillna(movies["runtime"].median(), inplace=True)
    movies["vote_average"].fillna(movies["vote_average"].median(), inplace=True)
    movies["vote_count"].fillna(movies["vote_count"].median(), inplace=True)
    movies.drop_duplicates(subset=["title"], inplace=True)

    movies["log_budget"]     = np.log1p(movies["budget"])
    movies["log_revenue"]    = np.log1p(movies["revenue"])
    movies["log_votes"]      = np.log1p(movies["vote_count"])
    movies["log_popularity"] = np.log1p(movies["popularity"])

    X = movies[FEATURES]
    y = movies[TARGET]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)

    pipe = Pipeline([
        ("sc", StandardScaler()),
        ("m",  XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                            subsample=0.8, colsample_bytree=0.8,
                            random_state=42, verbosity=0))
    ])
    pipe.fit(X_train, y_train)

    pred   = pipe.predict(X_test)
    r2     = r2_score(y_test, pred)
    mae    = mean_absolute_error(y_test, pred)

    # SHAP sobre muestra del test
    n_shap = min(600, len(X_test))
    idx    = np.random.choice(len(X_test), n_shap, replace=False)
    X_sc   = pd.DataFrame(pipe.named_steps["sc"].transform(X_test),
                          columns=FEATURES)
    X_shap = X_sc.iloc[idx].reset_index(drop=True)
    fv_shap = X_test.iloc[idx].reset_index(drop=True)

    explainer   = shap.TreeExplainer(pipe.named_steps["m"])
    shap_values = explainer.shap_values(X_shap)
    shap_df     = pd.DataFrame(shap_values, columns=FEATURES)
    mean_shap   = pd.Series(np.abs(shap_values).mean(axis=0), index=FEATURES)\
                    .sort_values(ascending=False)

    return pipe, r2, mae, mean_shap, shap_df, fv_shap, movies


print("⏳  Entrenando modelo XGBoost…")
PIPE, R2, MAE, MEAN_SHAP, SHAP_DF, FV_SHAP, MOVIES = entrenar_modelo()
print(f"✅  Modelo listo  |  R²={R2:.3f}  MAE={MAE:.3f}")



⏳  Entrenando modelo XGBoost…
✅  Modelo listo  |  R²=0.826  MAE=0.245


In [ ]:
# ════════════════════════════════════════════════════════════
#  HELPERS DE VISUALIZACIÓN
# ════════════════════════════════════════════════════════════

def nivel_popularidad(pop_log):
    pop = np.expm1(pop_log)
    if pop < 5:     return 0
    elif pop < 15:  return 1
    elif pop < 40:  return 2
    elif pop < 100: return 3
    elif pop < 300: return 4
    else:           return 5

def gauge_chart(pop_log):
    pop   = float(np.expm1(pop_log))
    nivel = nivel_popularidad(pop_log)
    label = NIVEL_LABELS[nivel]
    emoji = NIVEL_EMOJIS[nivel]
    pct   = min(pop / 500, 1.0)

    steps = [
        dict(range=[0,5],   color="#1e3a5f"),
        dict(range=[5,15],  color="#1a5276"),
        dict(range=[15,40], color="#1f618d"),
        dict(range=[40,100],color="#2874a6"),
        dict(range=[100,300],color="#2e86c1"),
        dict(range=[300,500],color="#3498db"),
    ]
    fig = go.Figure(go.Indicator(
        mode="gauge+number+delta",
        value=round(pop, 1),
        number=dict(suffix="  pts", font=dict(size=32, color="#ecf0f1")),
        title=dict(text=f"<b>{emoji} {label}</b><br><sup>Popularidad TMDB estimada</sup>",
                   font=dict(size=15, color="#bdc3c7")),
        gauge=dict(
            axis=dict(range=[0, 500], tickcolor="#7f8c8d",
                      tickfont=dict(color="#bdc3c7", size=10)),
            bar=dict(color="#3498db", thickness=0.25),
            bgcolor="#1a252f",
            bordercolor="#2c3e50",
            steps=steps,
            threshold=dict(
                line=dict(color="#e74c3c", width=3),
                thickness=0.8, value=pop
            )
        )
    ))
    fig.update_layout(
        height=260, margin=dict(t=60, b=20, l=30, r=30),
        paper_bgcolor="#1a252f", font=dict(color="#ecf0f1")
    )
    return fig

def shap_waterfall(input_vals):
    """Waterfall de contribuciones SHAP para la predicción actual."""
    x_in = pd.DataFrame([input_vals], columns=FEATURES)
    x_sc = pd.DataFrame(PIPE.named_steps["sc"].transform(x_in), columns=FEATURES)
    exp  = shap.TreeExplainer(PIPE.named_steps["m"])
    sv   = exp.shap_values(x_sc)[0]

    base  = float(exp.expected_value)
    pairs = sorted(zip(FEATURES, sv), key=lambda t: abs(t[1]), reverse=True)[:8]
    feats = [p[0] for p in pairs]
    vals  = [p[1] for p in pairs]

    colors = ["#2ecc71" if v > 0 else "#e74c3c" for v in vals]
    hover  = [f"{'↑' if v>0 else '↓'} {abs(v):.4f}" for v in vals]

    fig = go.Figure(go.Bar(
        x=vals, y=feats, orientation="h",
        marker_color=colors,
        text=hover, textposition="outside",
        textfont=dict(size=10, color="#bdc3c7"),
        hovertemplate="<b>%{y}</b><br>Contribución SHAP: %{x:.4f}<extra></extra>"
    ))
    fig.add_vline(x=0, line_color="#7f8c8d", line_width=1)
    fig.update_layout(
        title=dict(text="<b>¿Por qué esta predicción?</b> — Contribuciones SHAP",
                   font=dict(size=13, color="#ecf0f1")),
        height=320, margin=dict(t=50, b=20, l=20, r=60),
        paper_bgcolor="#1a252f", plot_bgcolor="#1e2d3d",
        xaxis=dict(gridcolor="#2c3e50", zerolinecolor="#7f8c8d",
                   tickfont=dict(color="#bdc3c7", size=10)),
        yaxis=dict(gridcolor="rgba(0,0,0,0)",
                   tickfont=dict(color="#ecf0f1", size=11))
    )
    return fig

def shap_global_bar():
    fig = go.Figure(go.Bar(
        x=MEAN_SHAP.values[::-1],
        y=MEAN_SHAP.index[::-1],
        orientation="h",
        marker=dict(
            color=list(range(len(MEAN_SHAP))),
            colorscale="Blues", showscale=False
        ),
        hovertemplate="<b>%{y}</b><br>|SHAP| medio: %{x:.4f}<extra></extra>"
    ))
    fig.update_layout(
        title=dict(text="<b>Importancia global de variables (SHAP)</b>",
                   font=dict(size=13, color="#ecf0f1")),
        height=340, margin=dict(t=50, b=20, l=20, r=20),
        paper_bgcolor="#1a252f", plot_bgcolor="#1e2d3d",
        xaxis=dict(title="mean |SHAP|", gridcolor="#2c3e50",
                   tickfont=dict(color="#bdc3c7", size=10)),
        yaxis=dict(gridcolor="rgba(0,0,0,0)",
                   tickfont=dict(color="#ecf0f1", size=11))
    )
    return fig

def scatter_comparativo(pop_pred_log):
    sample = MOVIES.sample(min(1500, len(MOVIES)), random_state=1)
    pop_pred = float(np.expm1(pop_pred_log))

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=sample["vote_count"].clip(upper=5000),
        y=np.expm1(sample["log_popularity"]).clip(upper=500),
        mode="markers",
        marker=dict(size=4, color="rgba(52,152,219,0.35)"),
        name="Dataset",
        hovertemplate="Votos: %{x:,.0f}<br>Popularidad: %{y:.1f}<extra></extra>"
    ))
    fig.add_trace(go.Scatter(
        x=[None], y=[pop_pred],
        mode="markers",
        marker=dict(size=16, color="#e74c3c", symbol="star",
                    line=dict(width=2, color="#fff")),
        name="Tu película",
    ))
    fig.add_hline(y=pop_pred, line_dash="dash",
                  line_color="#e74c3c", line_width=1.5,
                  annotation_text=f"  Tu película: {pop_pred:.1f}",
                  annotation_font_color="#e74c3c")

    fig.update_layout(
        title=dict(text="<b>Tu película vs el dataset</b>",
                   font=dict(size=13, color="#ecf0f1")),
        height=300, margin=dict(t=50, b=40, l=50, r=20),
        paper_bgcolor="#1a252f", plot_bgcolor="#1e2d3d",
        xaxis=dict(title="Votos (clip 5k)", gridcolor="#2c3e50",
                   tickfont=dict(color="#bdc3c7", size=10)),
        yaxis=dict(title="Popularidad TMDB", gridcolor="#2c3e50",
                   tickfont=dict(color="#bdc3c7", size=10)),
        legend=dict(bgcolor="rgba(0,0,0,0)", font=dict(color="#bdc3c7"))
    )
    return fig



In [ ]:
# ════════════════════════════════════════════════════════════
#  FUNCIÓN PRINCIPAL DE PREDICCIÓN
# ════════════════════════════════════════════════════════════

def predecir(budget, revenue, runtime, vote_average, vote_count,
             drama, comedy, thriller, action, romance, horror, crime, adventure):

    vals = {
        "log_budget":   np.log1p(float(budget)),
        "log_revenue":  np.log1p(float(revenue)),
        "runtime":      float(runtime),
        "vote_average": float(vote_average),
        "log_votes":    np.log1p(float(vote_count)),
        "Drama":    int(drama),
        "Comedy":   int(comedy),
        "Thriller": int(thriller),
        "Action":   int(action),
        "Romance":  int(romance),
        "Horror":   int(horror),
        "Crime":    int(crime),
        "Adventure":int(adventure),
    }

    x_in    = pd.DataFrame([[vals[f] for f in FEATURES]], columns=FEATURES)
    pop_log = float(PIPE.predict(x_in)[0])
    pop     = np.expm1(pop_log)
    nivel   = nivel_popularidad(pop_log)

    # Percentil aproximado en el dataset
    pct = int((MOVIES["log_popularity"] < pop_log).mean() * 100)

    # ── Insight textual ─────────────────────────────────────
    top_driver = MEAN_SHAP.index[0]
    x_sc = pd.DataFrame(PIPE.named_steps["sc"].transform(x_in), columns=FEATURES)
    sv   = shap.TreeExplainer(PIPE.named_steps["m"]).shap_values(x_sc)[0]
    shap_dict = dict(zip(FEATURES, sv))
    top_local = max(shap_dict, key=lambda k: abs(shap_dict[k]))

    insight = (
        f"**Popularidad estimada: {pop:.1f} pts** — {NIVEL_EMOJIS[nivel]} {NIVEL_LABELS[nivel]}\n\n"
        f"Tu película supera al **{pct}%** de las películas del dataset.\n\n"
        f"**Factor más influyente en esta predicción:** `{top_local}` "
        f"({'↑ empuja al alza' if shap_dict[top_local]>0 else '↓ reduce popularidad'})\n\n"
        f"**Consejo:** {'Incrementar la distribución (revenue) podría ser tu mayor palanca.' if top_local in ['log_revenue','log_budget'] else 'Generar más engagement temprano (votos/ratings) es la acción con mayor impacto según el modelo.'}"
    )

    input_vals = [vals[f] for f in FEATURES]

    return (
        gauge_chart(pop_log),
        shap_waterfall(input_vals),
        scatter_comparativo(pop_log),
        shap_global_bar(),
        insight
    )


# ════════════════════════════════════════════════════════════
#  INTERFAZ GRADIO
# ════════════════════════════════════════════════════════════

CSS = """
body, .gradio-container { background: #111827 !important; color: #e5e7eb !important; font-family: 'Inter', sans-serif; }
.gr-panel, .gr-box { background: #1f2937 !important; border: 1px solid #374151 !important; border-radius: 12px !important; }
h1 { color: #60a5fa !important; }
h3 { color: #93c5fd !important; }
label { color: #d1d5db !important; font-size: 13px !important; }
.gr-button-primary { background: #2563eb !important; border: none !important; border-radius: 8px !important; color: white !important; font-weight: 600 !important; }
.gr-button-primary:hover { background: #1d4ed8 !important; }
footer { display: none !important; }
.gr-markdown p { color: #d1d5db !important; line-height: 1.6; }
.gr-markdown strong { color: #60a5fa !important; }
.gr-markdown code { background: #374151; padding: 2px 6px; border-radius: 4px; color: #a5f3fc; }
"""

DESCRIPTION = """
## 🎬 Predictor de Popularidad de Películas — TMDB

**Pregunta del proyecto:** *¿Qué características de una película predicen mejor su popularidad en TMDB?*

Ingresa los datos de tu película y el modelo **XGBoost** (R²=0.74) te dará una estimación de popularidad
junto con una explicación SHAP de qué variables más influyeron en la predicción.
"""

with gr.Blocks(css=CSS, title="🎬 Movie Popularity Predictor") as demo:

    gr.Markdown(DESCRIPTION)

    with gr.Row():
        # ── Panel izquierdo: inputs ──────────────────────────
        with gr.Column(scale=1):
            gr.Markdown("### 💰 Datos económicos")
            budget = gr.Slider(
                0, 300_000_000, value=25_000_000, step=500_000,
                label="Presupuesto (USD)",
                info="Budget de producción en dólares"
            )
            revenue = gr.Slider(
                0, 2_000_000_000, value=80_000_000, step=1_000_000,
                label="Recaudación esperada (USD)",
                info="Revenue total estimado"
            )

            gr.Markdown("### 🎥 Características técnicas")
            runtime = gr.Slider(
                60, 240, value=105, step=1,
                label="Duración (minutos)"
            )
            vote_average = gr.Slider(
                1.0, 10.0, value=6.5, step=0.1,
                label="Calificación promedio esperada",
                info="Estimación de vote_average en TMDB"
            )
            vote_count = gr.Slider(
                0, 10_000, value=500, step=50,
                label="Número de votos esperados",
                info="Cuántas personas esperas que voten"
            )

            gr.Markdown("### 🎭 Géneros (selecciona todos los que apliquen)")
            with gr.Row():
                drama    = gr.Checkbox(label="Drama")
                comedy   = gr.Checkbox(label="Comedy")
                thriller = gr.Checkbox(label="Thriller")
                action   = gr.Checkbox(label="Action", value=True)
            with gr.Row():
                romance  = gr.Checkbox(label="Romance")
                horror   = gr.Checkbox(label="Horror")
                crime    = gr.Checkbox(label="Crime")
                adventure= gr.Checkbox(label="Adventure")

            btn = gr.Button("🔮 Predecir popularidad", variant="primary", size="lg")

        # ── Panel derecho: outputs ───────────────────────────
        with gr.Column(scale=2):
            gr.Markdown("### 📊 Resultado de la predicción")
            insight_md = gr.Markdown("*Ajusta los parámetros y presiona Predecir…*")
            gauge_out   = gr.Plot(label="Medidor de popularidad")

            with gr.Row():
                waterfall_out = gr.Plot(label="¿Por qué esta predicción? (SHAP)")
                scatter_out   = gr.Plot(label="Tu película vs el dataset")

            global_shap_out = gr.Plot(label="Importancia global de variables")

    # ── Botón de predicción ──────────────────────────────────
    btn.click(
        fn=predecir,
        inputs=[budget, revenue, runtime, vote_average, vote_count,
                drama, comedy, thriller, action, romance, horror, crime, adventure],
        outputs=[gauge_out, waterfall_out, scatter_out, global_shap_out, insight_md]
    )

    # ── Predicción automática al cargar ─────────────────────
    demo.load(
        fn=predecir,
        inputs=[budget, revenue, runtime, vote_average, vote_count,
                drama, comedy, thriller, action, romance, horror, crime, adventure],
        outputs=[gauge_out, waterfall_out, scatter_out, global_shap_out, insight_md]
    )

    gr.Markdown("""
---
**Notas del modelo:**
- Entrenado con ~40K películas de TMDB (hasta ~2017).
- La popularidad TMDB es dinámica; estos valores son estimaciones basadas en patrones históricos.
- El modelo explica el **74.2%** de la varianza en popularidad (R² en test set).
""")


if __name__ == "__main__":
    demo.launch(
        share=False,          # True para generar link público temporal
        server_port=7860,
        show_error=True
    )


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>


#  SECCIÓN 4 — CONCLUSIONES


In [ ]:
top3 = importancia.head(3).index.tolist()

print("=" * 62)
print("   SECCIÓN 4 — HALLAZGOS, LIMITACIONES Y RECOMENDACIÓN")
print("=" * 62)
print(f"""
¿QUÉ ENCONTRAMOS?
─────────────────
El modelo {mejor['model']} explica el {mejor['r2']*100:.1f}% de la varianza
en la popularidad (log) de películas TMDB.

Variables más determinantes (SHAP):
  1. {top3[0]}   → Motor principal de popularidad
  2. {top3[1]}  → Proxy de alcance de distribución
  3. {top3[2]}  → Escala y ambición de producción

La calidad (vote_average) tiene impacto sorprendentemente bajo,
lo que revela que la popularidad es un fenómeno de MASA, no de
calidad: se necesita llegar a mucha gente, no solo gustarles.

LIMITACIONES
────────────
  1. Budget con muchos ceros no reportados → ruido en log_budget.
  2. Popularidad TMDB es dinámica; valores históricos pueden estar
     desactualizados respecto al momento real de máxima visibilidad.
  3. Sin datos de elenco, director ni plataforma de distribución,
     variables que probablemente tienen alto poder predictivo.
  4. Dataset hasta ~2017; el streaming post-2019 cambió radicalmente
     los patrones de popularidad (series vs. películas).

RECOMENDACIÓN CONCRETA
──────────────────────
Para maximizar popularidad: invertir en DISTRIBUCIÓN y ENGAGEMENT
antes que en presupuesto de producción puro.

  ✅ Estrategia ganadora:
     Asegurar estreno simultáneo en múltiples mercados (→ revenue alto)
     + campaña activa de reseñas y ratings tempranos (→ vote_count alto)
     = efecto multiplicador según el modelo.

  ⚠️  Trampa a evitar:
     Producción costosa sin distribución amplia tiene menor ROI
     en popularidad que una producción modesta con distribución global.
""")


   SECCIÓN 4 — HALLAZGOS, LIMITACIONES Y RECOMENDACIÓN

¿QUÉ ENCONTRAMOS?
─────────────────
El modelo XGBoost explica el 82.8% de la varianza
en la popularidad (log) de películas TMDB.

Variables más determinantes (SHAP):
  1. log_votes   → Motor principal de popularidad
  2. log_budget  → Proxy de alcance de distribución
  3. vote_average  → Escala y ambición de producción

La calidad (vote_average) tiene impacto sorprendentemente bajo,
lo que revela que la popularidad es un fenómeno de MASA, no de
calidad: se necesita llegar a mucha gente, no solo gustarles.

LIMITACIONES
────────────
  1. Budget con muchos ceros no reportados → ruido en log_budget.
  2. Popularidad TMDB es dinámica; valores históricos pueden estar
     desactualizados respecto al momento real de máxima visibilidad.
  3. Sin datos de elenco, director ni plataforma de distribución,
     variables que probablemente tienen alto poder predictivo.
  4. Dataset hasta ~2017; el streaming post-2019 cambió radicalmente
     l


#Sección 5 — Uso de Inteligencia Artificial en el Desarrollo del Proyecto

¿Cómo usé IA en este proyecto?
A lo largo del desarrollo utilicé Claude (Anthropic) como asistente principal, integrándolo en cada etapa del flujo de trabajo. No fue un uso puntual para resolver un problema aislado, sino una colaboración continua que atravesó todas las fases del proyecto.

E**xploración y planteamiento (Sección 1)** **texto en negrita**
Le describí el dataset con su estructura (dtypes, valores faltantes, dimensiones) y le pedí que me ayudara a formular una pregunta analítica concreta y justificada. La IA propuso enfocar el problema en la predicción de popularidad como variable objetivo, argumentando por qué era más interesante que predecir vote_average o revenue. También sugirió la transformación logarítmica para las variables con sesgo fuerte, algo que yo había pasado por alto inicialmente.
Limpieza de datos
Compartí el output de df.info() y df.isnull().sum() y le pregunté cuál era la estrategia mínima necesaria de limpieza. La IA identificó que el presupuesto tenía ceros enmascarados como valores válidos (no como NaN) y recomendó tratarlos como faltantes antes de imputar, lo que mejoró la calidad del feature log_budget.

**Modelado y pipelines (Sección 2)**
Le pedí que comparara tres modelos apropiados para el problema, que generara los Pipelines de Scikit-learn y que eligiera métricas justificadas (MAE, RMSE, R² y CV-R² en lugar de solo accuracy). La IA además propuso el radar chart normalizado como forma más honesta de comparar modelos con métricas en escalas distintas, en lugar del clásico bar chart.

**Explicabilidad SHAP (Sección 3)**
Esta fue la sección donde más valor aportó la IA. Le describí los tres tipos de gráficos SHAP que quería y me propuso generar versiones interactivas en Plotly en lugar del shap.summary_plot estático por defecto. También construyó el dashboard HTML con Chart.js integrado directamente en la conversación, con beeswarm interactivo, dependence plot con selector de variables y cards de métricas, todo conectado a datos simulados que replican la distribución real del modelo.

**Mini app Gradio (Sección 4 adicional)**
Le expliqué la pregunta del proyecto y le pedí una app que la respondiera de forma interactiva. La IA estructuró la app con un panel de inputs organizado por bloques temáticos, cuatro visualizaciones Plotly que se actualizan en tiempo real con cada predicción, y un generador automático de insights textuales que conecta el resultado numérico con una recomendación accionable.

# **Reflexión crítica**
El uso de IA fue eficiente pero no pasivo. En cada iteración revisé el código generado, lo adapté al contexto real del dataset, corregí valores por defecto que no tenían sentido para datos de películas (por ejemplo, rangos de sliders), y tomé decisiones de diseño propias como qué variables incluir o qué géneros priorizar.
Lo más valioso no fue que la IA "hiciera el proyecto", sino que aceleró las decisiones técnicas: en lugar de investigar qué tipo de explainer SHAP usar para XGBoost, o cómo normalizar métricas heterogéneas para un radar chart, pude obtener una propuesta razonada en segundos y dedicar el tiempo a evaluar si tenía sentido para el problema específico.
Como limitación, la IA no tiene acceso al dataset real, por lo que los valores SHAP del dashboard son simulados con distribuciones representativas. Cualquier interpretación cuantitativa precisa requiere ejecutar el pipeline completo con los datos reales.